# WALMART SALES ANALYSIS (USING PYTHON ADN SQL)

## 📥 Loading the Dataset Directly from the Kaggle Source

In [83]:
pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [84]:
import kagglehub
import shutil
import os

# Download dataset
path = kagglehub.dataset_download(
    "najir0123/walmart-10k-sales-datasets"
)

# Destination folder
destination = "./datasets"

# Create data folder if it doesn't exist
os.makedirs(destination, exist_ok=True)

# Copy all downloaded files into data/
for file in os.listdir(path):
    source = os.path.join(path, file)
    target = os.path.join(destination, file)
    shutil.copy2(source, target)

print("Dataset copied to:", os.path.abspath(destination))

Dataset copied to: c:\Projects\SQL Projects\Walmart Sales Analysis\notebooks\datasets


## Importing Libraries

In [85]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 Loading the Dataset

The Walmart sales dataset is loaded from the local `datasets` directory for further data preparation and analysis.

In [86]:
df = pd.read_csv("../datasets/Walmart.csv")
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


## 🔍 Initial Dataset Overview

Before cleaning the dataset, its structure, dimensions, data types, and overall characteristics are examined to identify potential data quality issues.

In [87]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (10051, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  object 
 2   City            10051 non-null  object 
 3   category        10051 non-null  object 
 4   unit_price      10020 non-null  object 
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  object 
 7   time            10051 non-null  object 
 8   payment_method  10051 non-null  object 
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 863.9+ KB


In [88]:
df.describe(include='all')

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
count,10051.000000,10051,10051,10051,10020,10020.000000,10051,10051,10051,10051.000000,10051.000000
unique,NaN,100,98,6,1008,NaN,1460,1001,3,NaN,NaN
top,NaN,WALM058,Weslaco,Fashion accessories,$63,NaN,01/12/21,15:48:00,Credit card,NaN,NaN
freq,NaN,240,399,4579,159,NaN,48,33,4260,NaN,NaN
mean,5025.741220,NaN,NaN,NaN,NaN,2.353493,NaN,NaN,NaN,5.825659,0.393791
std,2901.174372,NaN,NaN,NaN,NaN,1.602658,NaN,NaN,NaN,1.763991,0.090669
min,1.000000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,3.000000,0.180000
25%,2513.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,4.000000,0.330000
50%,5026.000000,NaN,NaN,NaN,NaN,2.000000,NaN,NaN,NaN,6.000000,0.330000
75%,7538.500000,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,NaN,7.000000,0.480000


## 🧹 Checking for Missing Values and Handling them

Missing values are identified across all columns to determine whether any records require imputation, correction, or removal.

In [89]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

unit_price    31
quantity      31
dtype: int64

In [90]:
df[['unit_price','quantity',]].isna().mean()*100

unit_price    0.308427
quantity      0.308427
dtype: float64

In [91]:
### Since the missing values are less than 1% of the total records, we can choose to drop these rows without significantly impacting the dataset's integrity.
df = df.dropna(subset=['unit_price', 'quantity'])

### Now Checking if any null values are there

In [92]:
df.isnull().sum()

invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

## 🔄 Checking for Duplicate Records

Duplicate records are identified to prevent duplicated transactions from distorting sales, revenue, profitability, and customer metrics.

In [93]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 51


In [94]:
print("Duplicate Invoice IDs:", df["invoice_id"].duplicated().sum())

Duplicate Invoice IDs: 51


## 🗑️ Removing Duplicate Records

Exact duplicate records are removed to ensure that each transaction is represented only once in the analytical dataset.

In [95]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset Shape After Removing Duplicates:", df.shape)

Dataset Shape After Removing Duplicates: (9969, 11)


## 🏷️ Standardizing Column Names

Column names are standardized using lowercase and snake_case formatting to improve consistency and make the dataset easier to work with in Python and MySQL.

In [96]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

Index(['invoice_id', 'branch', 'city', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin'],
      dtype='object')

## 💰 Cleaning and Converting Unit Price

The `unit_price` column contains currency symbols and is therefore stored as a string. The currency symbol is removed and the column is converted to a numeric data type for financial calculations.

In [97]:
df["unit_price"].dtype

dtype('O')

In [98]:
df['unit_price'] = (df['unit_price'].str.replace("$", "", regex=False).str.strip().astype(float))

In [99]:
df['unit_price'].dtype

dtype('float64')

In [100]:
df['unit_price'].head()

0    74.69
1    15.28
2    46.33
3    58.22
4    86.31
Name: unit_price, dtype: float64

## 🔢 Converting Quantity to Integer

The quantity field represents the number of units sold per transaction. Since transaction quantities are whole numbers, the column is converted to an integer data type.

In [101]:
df['quantity']

0       7.0
1       5.0
2       7.0
3       8.0
4       7.0
       ... 
9964    3.0
9965    2.0
9966    3.0
9967    2.0
9968    3.0
Name: quantity, Length: 9969, dtype: float64

In [102]:
df["quantity"] = df["quantity"].astype(int)

## 📅 Converting Transaction Date

The transaction date is converted from text into a proper datetime format to enable time-based feature engineering and subsequent sales analysis in MySQL.

In [103]:
df['date']

0       05/01/19
1       08/03/19
2       03/03/19
3       27/01/19
4       08/02/19
          ...   
9964    03/08/23
9965    22/02/21
9966    15/06/23
9967    25/02/21
9968    26/09/20
Name: date, Length: 9969, dtype: object

In [104]:
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%y')

In [105]:
df['date'].dtype

dtype('<M8[ns]')

## ⏰ Converting Transaction Time

The transaction time is converted into a datetime-compatible time representation, allowing hour-level and time-of-day analysis.

In [106]:
df['time']

0       13:08:00
1       10:29:00
2       13:23:00
3       20:33:00
4       10:37:00
          ...   
9964    10:10:00
9965    14:20:00
9966    16:00:00
9967    12:25:00
9968     9:48:00
Name: time, Length: 9969, dtype: object

In [107]:
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S')

In [108]:
df['time'].dtype

dtype('<M8[ns]')

## 🧽 Standardizing Categorical Fields

Categorical columns are cleaned by removing unnecessary whitespace and standardizing text formatting to ensure consistent grouping and filtering during SQL analysis.

In [109]:
text_columns = ["branch","city","category","payment_method"]

for col in text_columns:
    df[col] = df[col].str.strip()

## ✅ Validating Numerical Values

Numerical fields are checked for invalid or unrealistic values to ensure that the dataset satisfies basic business and analytical constraints.

In [110]:
print("Invalid Unit Prices:", (df["unit_price"] <= 0).sum())
print("Invalid Quantities:", (df["quantity"] <= 0).sum())
print("Invalid Ratings:", ((df["rating"] < 0) | (df["rating"] > 10)).sum())
print("Invalid Profit Margins:", ((df["profit_margin"] < 0) | (df["profit_margin"] > 1)).sum())

Invalid Unit Prices: 0
Invalid Quantities: 0
Invalid Ratings: 0
Invalid Profit Margins: 0


## 🚀 Feature Engineering

In [111]:
df.head()

,invoice_id,branch,city,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,74.69,7,2019-01-05,1900-01-01 13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,15.28,5,2019-03-08,1900-01-01 10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7,2019-03-03,1900-01-01 13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,58.22,8,2019-01-27,1900-01-01 20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,86.31,7,2019-02-08,1900-01-01 10:37:00,Ewallet,5.3,0.48


## 💵 Creating Total Sales

A `total_sales` feature is created by multiplying the unit price by the quantity sold. This represents the total revenue generated from each transaction.

In [112]:
df['total_sales'] = df['unit_price'] * df['quantity']

In [113]:
df.head(2)

,invoice_id,branch,city,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total_sales
0,1,WALM003,San Antonio,Health and beauty,74.69,7,2019-01-05,1900-01-01 13:08:00,Ewallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5,2019-03-08,1900-01-01 10:29:00,Cash,9.6,0.48,76.40


## 📈 Creating Gross Profit

A `gross_profit` feature is calculated using total sales and the corresponding profit margin. This provides a transaction-level measure of estimated profit for profitability analysis.

In [114]:
df['gross_profit'] = df['total_sales'] * df['profit_margin']
df.head(2)

,invoice_id,branch,city,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total_sales,gross_profit
0,1,WALM003,San Antonio,Health and beauty,74.69,7,2019-01-05,1900-01-01 13:08:00,Ewallet,9.1,0.48,522.83,250.9584
1,2,WALM048,Harlingen,Electronic accessories,15.28,5,2019-03-08,1900-01-01 10:29:00,Cash,9.6,0.48,76.40,36.6720


## 📆 Extracting Year

The transaction year is extracted from the date to support year-over-year sales and profitability analysis.

In [115]:
df["year"] = df["date"].dt.year

## 📅 Extracting Month

The transaction month is extracted to enable monthly sales, revenue, and profitability analysis.

In [116]:
df["month"] = df["date"].dt.month

## 🗓️ Creating Month Name

The month name is derived from the transaction date to make time-based reporting more readable and business-friendly.

In [117]:
df["month_name"] = df["date"].dt.month_name()

## 📊 Creating Quarter

A quarterly feature is created to support seasonal and quarterly sales performance analysis.

In [118]:
df["quarter"] = df["date"].dt.quarter

## 📆 Creating Day of Week

The day of the week is extracted to analyze customer purchasing patterns and identify high-performing weekdays.

In [119]:
df["day_of_week"] = df["date"].dt.day_name()

## 🛍️ Creating Weekend Indicator

A binary `is_weekend` feature is created to distinguish weekend transactions from weekday transactions and support comparative sales analysis.

In [120]:
df["is_weekend"] = df["date"].dt.dayofweek >= 5

df["is_weekend"] = df["is_weekend"].astype(int)

## ⏰ Extracting Transaction Hour

The transaction hour is extracted from the time field to enable hourly sales and customer purchasing pattern analysis.

In [121]:
df["hour"] = pd.to_datetime(df["time"].astype(str)).dt.hour

## ⭐ Creating Customer Rating Category

Customer ratings are categorized into meaningful satisfaction levels to simplify customer experience analysis.

In [122]:
def rating_category(rating):
    if rating >= 8:
        return "Excellent"
    elif rating >= 6:
        return "Good"
    elif rating >= 4:
        return "Average"
    else:
        return "Poor"

df["rating_category"] = df["rating"].apply(rating_category)

## 🔎 Final Data Quality Check

The cleaned and feature-engineered dataset is reviewed to verify its structure, data types, missing values, duplicate records, and newly created analytical features.

In [123]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9969 entries, 0 to 9968
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   invoice_id       9969 non-null   int64         
 1   branch           9969 non-null   object        
 2   city             9969 non-null   object        
 3   category         9969 non-null   object        
 4   unit_price       9969 non-null   float64       
 5   quantity         9969 non-null   int64         
 6   date             9969 non-null   datetime64[ns]
 7   time             9969 non-null   datetime64[ns]
 8   payment_method   9969 non-null   object        
 9   rating           9969 non-null   float64       
 10  profit_margin    9969 non-null   float64       
 11  total_sales      9969 non-null   float64       
 12  gross_profit     9969 non-null   float64       
 13  year             9969 non-null   int32         
 14  month            9969 non-null   int32  

In [124]:
df.duplicated().sum()

np.int64(0)

## 📋 Final Analytical Dataset

The final dataset contains cleaned transactional attributes along with derived sales, profitability, date, time, and customer-experience features prepared for downstream SQL analysis.

In [126]:
final_columns = [
    "invoice_id",
    "branch",
    "city",
    "category",
    "unit_price",
    "quantity",
    "total_sales",
    "profit_margin",
    "gross_profit",
    "date",
    "year",
    "month",
    "month_name",
    "quarter",
    "day_of_week",
    "is_weekend",
    "time",
    "hour",
    "payment_method",
    "rating",
    "rating_category"
]

df = df[final_columns]

In [127]:
df.head()

,invoice_id,branch,city,category,unit_price,quantity,total_sales,profit_margin,gross_profit,date,...,month,month_name,quarter,day_of_week,is_weekend,time,hour,payment_method,rating,rating_category
0,1,WALM003,San Antonio,Health and beauty,74.69,7,522.83,0.48,250.9584,2019-01-05,...,1,January,1,Saturday,1,1900-01-01 13:08:00,13,Ewallet,9.1,Excellent
1,2,WALM048,Harlingen,Electronic accessories,15.28,5,76.40,0.48,36.6720,2019-03-08,...,3,March,1,Friday,0,1900-01-01 10:29:00,10,Cash,9.6,Excellent
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7,324.31,0.33,107.0223,2019-03-03,...,3,March,1,Sunday,1,1900-01-01 13:23:00,13,Credit card,7.4,Good
3,4,WALM064,Bedford,Health and beauty,58.22,8,465.76,0.33,153.7008,2019-01-27,...,1,January,1,Sunday,1,1900-01-01 20:33:00,20,Ewallet,8.4,Excellent
4,5,WALM013,Irving,Sports and travel,86.31,7,604.17,0.48,290.0016,2019-02-08,...,2,February,1,Friday,0,1900-01-01 10:37:00,10,Ewallet,5.3,Average


## 💾 Exporting the Cleaned Dataset for MySQL

The cleaned and feature-engineered dataset is exported as a CSV file. This version will serve as the source file for importing the prepared data into MySQL for business analysis.

In [128]:
df.to_csv("../datasets/walmart_sales_cleaned.csv",index=False)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.
